# 🎯 Train Mới Hoàn Toàn `YOLO11s` (`yolo11s.pt` → `yolo_best_1class.onnx`) — Bản Vẽ Lớn (Full-Page CAD)
**Mục tiêu:** Khởi tạo **hoàn toàn mới** từ trọng số gốc **`yolo11s.pt` (YOLO11 Small mới nhất, không fine-tune lại `best.pt` cũ)** ở độ phân giải **`1024x1024`**, 1-class (`0: dimension_callout`) để phát hiện chính xác **Blue-Box tổng** trên bản vẽ CAD lớn:
- **41 bản vẽ thực tế đã duyệt (42 trang, 2,005 Blue-Boxes)** + **64 bản vẽ Full-Page Synthesize (3,782 Blue-Boxes)**:
  - Tổng cộng **106 trang bản vẽ lớn (5,787 Blue-Boxes)**: `train = 100 trang`, `val = 6 trang bản vẽ thực tế 100%`.
  - Bao trọn cả cụm **Nominal + Stacked Tolerance** (`12.5 +0.02/-0.01`, `Ø18.5 +0.05/0`) không bị cắt lẹm phần dung sai.
  - Bắt nhạy các kích thước nhỏ/khó: `C0.02`, `R0.05`, `4-R0.05MAX`, góc DMS `30°15'20"`, kích thước gạch chân, chữ nghiêng/dọc (`90°/180°/270°/tilt`).

Sau khi train xong, Notebook sẽ tự động xuất ra:
1. **`yolo_best_1class.onnx`** (để bỏ thẳng vào `mini-pdf-ocr-tester/models/onnx_yolo/`).
2. **`best_fullpage_yolo11s.pt`** (để lưu vào `mini-pdf-ocr-tester/models/pt_library/`).

In [ ]:
# ==============================================================================
# BƯỚC 1: KIỂM TRA GPU & CÀI ĐẶT ULTRALYTICS + ONNX
# ==============================================================================
!nvidia-smi
!pip install -q ultralytics onnx onnxruntime-gpu
import torch, ultralytics
ultralytics.checks()

In [ ]:
# ==============================================================================
# BƯỚC 2: UPLOAD & GIẢI NÉN GÓI DATASET `yolo_fullpage_v1_finetune_pack.zip`
# ==============================================================================
import os, shutil, zipfile, yaml
from pathlib import Path

PACK_NAME = 'yolo_fullpage_v1_finetune_pack.zip'
if not os.path.exists(PACK_NAME):
    from google.colab import files
    print(f'🔼 Vui lòng chọn file {PACK_NAME} từ thư mục mini-pdf-ocr-tester...')
    uploaded = files.upload()
    PACK_NAME = list(uploaded.keys())[0]

work_dir = Path('/content/fullpage_work')
if work_dir.exists():
    shutil.rmtree(work_dir)
work_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(PACK_NAME, 'r') as zf:
    zf.extractall(work_dir)

ds_dir = (work_dir / 'yolo_dataset').resolve()
DATA_YAML = ds_dir / 'data.yaml'
DATA_YAML.write_text(
    yaml.safe_dump({
        'path': ds_dir.as_posix(),
        'train': 'train/images',
        'val': 'val/images',
        'names': {0: 'dimension_callout'},
    }, sort_keys=False),
    encoding='utf-8'
)
train_cnt = len(list((ds_dir / 'train/images').glob('*.jpg')))
val_cnt = len(list((ds_dir / 'val/images').glob('*.jpg')))
print(f'✅ Giải nén thành công! Train: {train_cnt} trang bản vẽ lớn | Val (100% thực tế): {val_cnt} trang')

In [ ]:
# ==============================================================================
# BƯỚC 3: TRAIN MỚI HOÀN TOÀN TRÊN `yolo11s.pt` (IMGSZ=1024, 1-CLASS BLUE-BOX)
# ==============================================================================
import subprocess, sys
try:
    import ultralytics, onnx
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics', 'onnx', 'onnxruntime'])

import torch
from ultralytics import YOLO

device = 0 if torch.cuda.is_available() else 'cpu'
BASE_MODEL = 'yolo11s.pt'  # Khởi tạo hoàn toàn mới từ YOLO11s gốc, không dùng best.pt cũ
print('✔ Khởi tạo model mới từ gốc:', BASE_MODEL, '| Thiết bị:', 'GPU CUDA' if device == 0 else 'CPU')

model = YOLO(BASE_MODEL)
results = model.train(
    data='/content/fullpage_work/yolo_dataset/data.yaml',
    epochs=90,
    patience=25,
    imgsz=1024,
    batch=8,
    device=device,
    pretrained=True,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.05,
    cos_lr=True,
    box=9.0,
    cls=1.0,
    hsv_h=0.0,
    hsv_s=0.1,
    hsv_v=0.25,
    degrees=2.0,
    translate=0.05,
    scale=0.30,
    fliplr=0.0,         # Tuyệt đối không lật ngang chữ số CAD
    flipud=0.0,         # Tuyệt đối không lật dọc
    mosaic=0.5,
    close_mosaic=20,
    project='/content/runs_fullpage_yolo11s',
    name='cad_fullpage_yolo11s_1class',
    exist_ok=True,
)
print('✅ Huấn luyện YOLO11s hoàn tất!')

In [ ]:
# ==============================================================================
# BƯỚC 4: EXPORT SANG ONNX CHUẨN (`imgsz=1024, simplify=False, opset=12, nms=False`)
# & TẢI VỀ `yolo_best_1class.onnx` + `best_fullpage_yolo11s.pt`
# ==============================================================================
import shutil
from pathlib import Path
from ultralytics import YOLO

candidates = []
if 'results' in globals() and hasattr(results, 'save_dir'):
    candidates.append(Path(results.save_dir) / 'weights' / 'best.pt')
if 'model' in globals() and hasattr(getattr(model, 'trainer', None), 'best'):
    candidates.append(Path(model.trainer.best))
candidates.append(Path('/content/runs_fullpage_yolo11s/cad_fullpage_yolo11s_1class/weights/best.pt'))
candidates.extend(sorted(Path('/content').rglob('**/cad_fullpage_yolo11s_1class*/weights/best.pt'), key=lambda p: p.stat().st_mtime, reverse=True))

best_ckpt = next((p for p in candidates if p and p.is_file()), None)
assert best_ckpt is not None, 'Không tìm thấy best.pt! Hãy kiểm tra Bước 3 đã chạy xong chưa.'
print('✔ Tìm thấy model đã train tại:', best_ckpt.resolve())

export_model = YOLO(str(best_ckpt))
onnx_path = Path(export_model.export(
    format='onnx',
    imgsz=1024,
    simplify=False,
    opset=12,
    nms=False,
))

out_onnx = Path('/content/yolo_best_1class.onnx')
out_pt = Path('/content/best_fullpage_yolo11s.pt')
shutil.copy2(onnx_path, out_onnx)
shutil.copy2(best_ckpt, out_pt)

print('====================================================')
print(f'✅ Đã xuất file .ONNX mới: {out_onnx} ({out_onnx.stat().st_size / (1024*1024):.2f} MB)')
print(f'✅ Đã xuất file .PT mới  : {out_pt} ({out_pt.stat().st_size / (1024*1024):.2f} MB)')
print('====================================================')

try:
    from google.colab import files
    files.download(str(out_onnx))
    files.download(str(out_pt))
except Exception:
    print('Đang chạy local — lấy trực tiếp 2 file yolo_best_1class.onnx và best_fullpage_yolo11s.pt.')